# A small convolutional network on Imagenette

**COMP3620 Neural Networks and Deep Learning — Coursework 2, notebook 3.**
Understanding Deep Learning (UDL), Chapter 10, with the training methods of
Chapters 6 and 7.

This is Part 4 of Coursework 2. In notebook 2 (`COMP3620_CW2_2_logistic_regression_pytorch.ipynb`) you
trained logistic regression on Imagenette. Here you train a small **convolutional network**
on the same images, with the same split into training and test set, and
compare. The worksheet explains why the network is built the way it is, and
which variations you could try.

Cells marked **TODO** are yours to complete; until you do, the runs that
need them print a reminder and are skipped, so the notebook always runs
from top to bottom.

Run it on a GPU: one 15-epoch run takes about two minutes there. The
Imagenette data and its cached copy are shared with notebook 2.

In [ ]:
import pathlib, time
import numpy as np
import matplotlib.pyplot as plt
import torch, torch.nn as nn
from torchvision import datasets, transforms
from PIL import Image

device = ('cuda' if torch.cuda.is_available() else
          'mps' if torch.backends.mps.is_available() else 'cpu')
print('torch', torch.__version__, '| device:', device)

DATA_ROOT = pathlib.Path('./data')   # shared with notebook 2
IMG_SIZE = 128

## 1. Data

The same images and the same 80/20 split as in notebook 2. This time the
images keep their shape, $3 \times 128 \times 128$, because a convolutional
network uses the arrangement of the pixels. Each colour channel is
standardised with the mean and standard deviation of that channel over the
**training set**.

In [ ]:
CLASS_NAMES = ['tench', 'English springer', 'cassette player', 'chain saw',
               'church', 'French horn', 'garbage truck', 'gas pump',
               'golf ball', 'parachute']


def load_images(root, size):
    """All Imagenette images (train + val folders), resized and centre-cropped
    to size x size, as uint8 tensors. Cached in root/imagenette2_<size>.pt."""
    cache = root / f'imagenette2_{size}.pt'
    if cache.exists():
        d = torch.load(cache)
        return d['X'], d['y']
    if not (root / 'imagenette2').exists():
        datasets.Imagenette(root, split='train', size='full', download=True)
    folders = root / 'imagenette2'
    wnids = sorted(p.name for p in (folders / 'train').iterdir() if p.is_dir())
    tf = transforms.Compose([transforms.Resize(size), transforms.CenterCrop(size),
                             transforms.PILToTensor()])
    files = [(f, c) for split in ('train', 'val') for c, w in enumerate(wnids)
             for f in sorted((folders / split / w).glob('*.JPEG'))]
    X = torch.empty(len(files), 3, size, size, dtype=torch.uint8)
    y = torch.empty(len(files), dtype=torch.long)
    t0 = time.time()
    for i, (f, c) in enumerate(files):
        X[i] = tf(Image.open(f).convert('RGB'))
        y[i] = c
        if (i + 1) % 2000 == 0:
            print(f'  decoded {i + 1}/{len(files)} images ({time.time() - t0:.0f} s)')
    torch.save({'X': X, 'y': y}, cache)
    return X, y


images, labels = load_images(DATA_ROOT, IMG_SIZE)
perm = torch.from_numpy(np.random.default_rng(0).permutation(len(images)))   # as notebook 2
images, labels = images[perm], labels[perm]
n_train = int(0.8 * len(images))
X_train, X_test = images[:n_train], images[n_train:]          # uint8, N x 3 x 128 x 128
y_train, y_test = labels[:n_train], labels[n_train:]

# Per-channel mean and standard deviation over the training set.
s = torch.zeros(3, dtype=torch.float64); s2 = torch.zeros(3, dtype=torch.float64)
for i in range(0, n_train, 1000):
    xb = X_train[i:i + 1000].double() / 255
    s += xb.sum(dim=(0, 2, 3)); s2 += (xb ** 2).sum(dim=(0, 2, 3))
n = n_train * IMG_SIZE * IMG_SIZE
MEAN = (s / n).float().view(1, 3, 1, 1)
STD = ((s2 / n - (s / n) ** 2).sqrt()).float().view(1, 3, 1, 1)
print(f'training: {len(X_train)} images   test: {len(X_test)} images')
print('channel means', MEAN.flatten().numpy().round(3), ' standard deviations', STD.flatten().numpy().round(3))


def to_input(xb):
    """uint8 batch -> standardised float batch on the device."""
    x = xb.to(device).float() / 255
    return (x - MEAN.to(device)) / STD.to(device)

## 2. Optimisers

**TODO.** Replace the cell below by your completed `make_optimiser` and
`make_scheduler` from notebook 2 (Section 2.3). The reference run in
Section 4 uses Adam with a learning rate that is halved every 5 epochs.

In [ ]:
# TODO: paste your completed make_optimiser and make_scheduler from notebook 2 here.
def make_optimiser(model, method, lr, momentum=0.9):
    if method == 'sgd':
        return torch.optim.SGD(model.parameters(), lr=lr)
    raise NotImplementedError(f"method={method!r}: copy your make_optimiser from notebook 2")


def make_scheduler(optimiser, lr_step):
    if lr_step == 0:
        return None
    raise NotImplementedError('copy your make_scheduler from notebook 2')

## 3. The network

The network is described by a list of **blocks**. Each block is a
convolution with `out` output channels, kernel size `k`, stride `s` and
padding `p`, followed by a ReLU and a max-pool of size `pool` (`pool=1`: no
pooling). After the last block, the head turns the feature maps into the 10
class scores: `head='gap'` averages each map over all positions (global
average pooling) and applies one linear layer; `head='flatten'` flattens
the maps into one vector and applies one linear layer. The softmax is in
the loss, as before.

`trace` shows, for every layer, the spatial size of the output,

$$H_\text{out}=\left\lfloor \frac{H_\text{in}+2P-K}{S}\right\rfloor + 1,$$

the receptive field (how many input pixels across one output unit sees,
UDL Section 10.2.6) and the number of parameters so far.

In [ ]:
class ConvNet(nn.Module):
    def __init__(self, config, head='gap', in_ch=3, n_classes=10):
        super().__init__()
        layers, c = [], in_ch
        for b in config:
            layers += [nn.Conv2d(c, b['out'], b['k'], b['s'], b['p']), nn.ReLU()]
            if b.get('pool', 1) > 1:
                layers.append(nn.MaxPool2d(b['pool']))
            c = b['out']
        self.features = nn.Sequential(*layers)
        if head == 'gap':
            self.head = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(c, n_classes))
        else:
            self.head = nn.Sequential(nn.Flatten(), nn.LazyLinear(n_classes))

    def forward(self, x):
        return self.head(self.features(x))


def trace(config, head='gap', img_size=IMG_SIZE, in_ch=3):
    """Output size, receptive field and parameter count after every layer."""
    size, rf, jump, params, c = img_size, 1, 1, 0, in_ch
    print(f"{'layer':<12}{'K/S/P':<9}{'output':<15}{'RF':>5}{'parameters':>13}")
    print(f"{'input':<12}{'':<9}{f'{c} x {size} x {size}':<15}{rf:>5}{0:>13,}")
    for i, b in enumerate(config):
        k, st, p = b['k'], b['s'], b['p']
        size = (size + 2 * p - k) // st + 1
        rf += (k - 1) * jump; jump *= st
        params += (c * k * k + 1) * b['out']; c = b['out']
        print(f"{f'conv{i + 1}':<12}{f'{k}/{st}/{p}':<9}"
              f"{f'{c} x {size} x {size}':<15}{rf:>5}{params:>13,}")
        if b.get('pool', 1) > 1:
            q = b['pool']
            size = (size - q) // q + 1
            rf += (q - 1) * jump; jump *= q
            print(f"{'  max-pool':<12}{f'{q}/{q}/0':<9}{f'{c} x {size} x {size}':<15}{rf:>5}{params:>13,}")
    n_in = c if head == 'gap' else c * size * size
    params += (n_in + 1) * 10
    print(f"{'head (' + head + ')':<12}{'':<9}{'10 scores':<15}{'':>5}{params:>13,}")
    return params


# The reference network: four blocks.
CNN4 = [
    dict(out=32,  k=7, s=2, p=3, pool=2),   # 128 -> 64 -> 32
    dict(out=64,  k=3, s=1, p=1, pool=2),   #  32 -> 32 -> 16
    dict(out=128, k=3, s=1, p=1, pool=2),   #  16 -> 16 ->  8
    dict(out=256, k=3, s=1, p=1, pool=2),   #   8 ->  8 ->  4
]
trace(CNN4);

## 4. Training and evaluation

`train` builds the network from a config with a fixed seed (PyTorch's
default initialisation), trains it, and then evaluates it once on the
training set and once on the test set; both are recorded in `RESULTS`. The
reference settings are Adam with a learning rate of 0.001, halved every 5
epochs, mini-batches of 128 and 15 epochs.

In [ ]:
RESULTS = []      # one row per run
REFERENCE = dict(method='adam', lr=1e-3, lr_step=5, epochs=15, batch_size=128)


@torch.no_grad()
def evaluate(model, X, y, bs=256):
    """Accuracy and mean cross-entropy loss of model on (X, y)."""
    model.eval()
    loss_fn = nn.CrossEntropyLoss(reduction='sum')
    total, correct = 0.0, 0
    for i in range(0, len(X), bs):
        scores = model(to_input(X[i:i + bs]))
        yb = y[i:i + bs].to(device)
        total += loss_fn(scores, yb).item()
        correct += (scores.argmax(1) == yb).sum().item()
    return correct / len(X), total / len(X)


def train(config, name, head='gap', method='adam', lr=1e-3, lr_step=5, epochs=15,
          batch_size=128, seed=0):
    """Train ConvNet(config, head); return the model and its history, or None
    if the method or schedule is not implemented yet."""
    torch.manual_seed(seed)
    model = ConvNet(config, head)
    model(torch.zeros(1, 3, IMG_SIZE, IMG_SIZE))          # sizes the lazy layer, if any
    model = model.to(device)
    n_params = sum(p.numel() for p in model.parameters())
    settings = (f'{name}: {method}, lr {lr:g}'
                + (f', halved every {lr_step}' if lr_step else ', fixed')
                + f', batch {batch_size}, {epochs} epochs')
    try:
        optimiser = make_optimiser(model, method, lr)
        scheduler = make_scheduler(optimiser, lr_step)
    except NotImplementedError as e:
        print(f'SKIPPED ({settings}): {e}.')
        return None
    loss_fn = nn.CrossEntropyLoss()
    history = {'loss': [], 'acc': []}
    t0 = time.time()
    for epoch in range(epochs):
        model.train()
        order = torch.randperm(len(X_train))
        running, correct = 0.0, 0
        for i in range(0, len(order), batch_size):
            idx = order[i:i + batch_size]
            xb, yb = to_input(X_train[idx]), y_train[idx].to(device)
            scores = model(xb)
            loss = loss_fn(scores, yb)
            optimiser.zero_grad()
            loss.backward()
            optimiser.step()
            running += loss.item() * len(idx)
            correct += (scores.argmax(1) == yb).sum().item()
        if scheduler is not None:
            scheduler.step()
        history['loss'].append(running / len(X_train))
        history['acc'].append(correct / len(X_train))
        print(f'  epoch {epoch + 1:2d}  training loss {history["loss"][-1]:.3f}  '
              f'training accuracy {history["acc"][-1]:.3f}  ({time.time() - t0:.0f} s)')
    tr_acc, tr_loss = evaluate(model, X_train, y_train)
    te_acc, te_loss = evaluate(model, X_test, y_test)
    print(f'{settings}  [{n_params:,} parameters]\n  training accuracy {tr_acc:.3f} '
          f'(loss {tr_loss:.3f})   test accuracy {te_acc:.3f} (loss {te_loss:.3f})')
    RESULTS.append(dict(settings=settings, params=n_params, train_acc=tr_acc,
                        test_acc=te_acc))
    return model, history


def plot_history(history, title):
    """Training loss and training accuracy per epoch (averaged over the epoch)."""
    ep = range(1, len(history['loss']) + 1)
    fig, ax = plt.subplots(1, 2, figsize=(10, 3.2))
    ax[0].plot(ep, history['loss']); ax[0].set_title('training loss')
    ax[1].plot(ep, history['acc']); ax[1].set_title('training accuracy'); ax[1].set_ylim(0, 1)
    ax[1].axhline(0.1, ls=':', c='gray', label='chance'); ax[1].legend()
    for a in ax: a.set_xlabel('epoch')
    fig.suptitle(title); plt.tight_layout(); plt.show()

### 4.1 The reference network

In [ ]:
out = train(CNN4, 'four blocks', **REFERENCE)
if out is not None:
    cnn4, hist = out
    plot_history(hist, 'Four blocks, reference settings')

The 32 filters of the first convolution, each $7 \times 7$ pixels in three
colours, shown as small images. Compare them with the templates of
logistic regression in notebook 2.

In [ ]:
if out is not None:
    W = cnn4.features[0].weight.detach().cpu()
    W = (W - W.amin(dim=(1, 2, 3), keepdim=True)) / (
        W.amax(dim=(1, 2, 3), keepdim=True) - W.amin(dim=(1, 2, 3), keepdim=True))
    fig, axes = plt.subplots(2, 16, figsize=(14, 2))
    for w, ax in zip(W, axes.flat):
        ax.imshow(w.permute(1, 2, 0)); ax.axis('off')
    fig.suptitle('First-layer filters'); plt.show()

## 5. Variations

The worksheet lists variations of the network and of its training. Write a
variation as a config (copy `CNN4` and change it), with the settings of
`train`; run `trace` on it first, to see what changes before you train.
Put your variations in `MY_VARIATIONS` and run the cell.

In [ ]:
# TODO: your variations. Each entry is (config, name, settings for train).
CNN3 = CNN4[:3]                                   # example: the first three blocks only
MY_VARIATIONS = [
    # (CNN3, 'three blocks', dict(REFERENCE)),
    # (CNN4, 'flatten head', dict(REFERENCE, head='flatten')),   # head is a setting of train
]

for config, name, settings in MY_VARIATIONS:
    print(name); trace(config, settings.get('head', 'gap'))
    train(config, name, **settings)

## 6. All your runs

In [ ]:
print(f"{'settings':<82}{'parameters':>12}{'train acc':>10}{'test acc':>10}")
for r in RESULTS:
    print(f"{r['settings']:<82}{r['params']:>12,}{r['train_acc']:>10.3f}{r['test_acc']:>10.3f}")